# VieISL MSKA++ ISLR Scratch

This version uses the corrected MSKA++ spatial attention, confidence-aware pooling, stream gates, and multi-scale temporal encoding. It is intended as a stronger ISLR encoder candidate for later CSLR transfer.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import csv
import json
import math
import os
import random
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch:", torch.__version__)
print("Torch CUDA runtime:", torch.version.cuda)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.10.0+cu128
Torch CUDA runtime: 12.8
Device: cuda
GPU: Tesla T4


In [2]:
# Run mode:
#   full : use the complete VieISL skeleton dataset.
#   smoke: use a small subset for quick path/model checks.
RUN_MODE = "full"

EXPERIMENT_NAME = "vieisl_mska_plus_islr_scratch"
MODEL_NAME = "mska_plus_islr"
DATASET_NAME = "VieISL"
TASK_NAME = "isolated_sign_recognition"

DATA_DIR = Path(r"/kaggle/input/datasets/leothn/vieisl/skeleton")
META_PATH = DATA_DIR / "dataset_meta.json"
OUTPUT_DIR = Path("/kaggle/working") / EXPERIMENT_NAME
EXPECTED_CLASSES = 216

CONFIG = {
    "seed": 491,
    "batch_size": 10,
    "epochs": 180,
    "lr": 2e-4,
    "weight_decay": 7e-4,
    "warmup_epochs": 10,
    "patience": 32,
    "grad_clip": 0.75,
    "num_workers": 2,
    "hidden": 256,
    "dropout": 0.35,
    "label_smoothing": 0.10,
    "use_class_weights": False,
    "use_weighted_sampler": False,
    "root_normalize": True,
    "time_stretch_prob": 0.65,
    "time_stretch_range": (0.82, 1.18),
    "spatial_jitter_std": 0.010,
    "confidence_dropout_prob": 0.15,
    "confidence_dropout_ratio": 0.06,
    "enable_horizontal_flip": False,
    "horizontal_flip_prob": 0.0,
    "use_motion": True,
    "use_conf_gate": True,
    "num_blocks": 4,
    "num_heads": 8,
    "temporal_blocks": 3,
    "temporal_kernel_sizes": (3, 5, 7),
    "temporal_rnn_layers": 2,
    "smoke_train_samples": 96,
    "smoke_eval_samples": 48,
    "smoke_epochs": 2,
}

if RUN_MODE == "smoke":
    CONFIG["epochs"] = CONFIG["smoke_epochs"]
    CONFIG["patience"] = CONFIG["smoke_epochs"]
    CONFIG["num_workers"] = 0

print("Experiment:", EXPERIMENT_NAME)
print("Model:", MODEL_NAME)
print("Run mode:", RUN_MODE)
print("Data dir:", DATA_DIR)
print("Metadata:", META_PATH)
print("Output dir:", OUTPUT_DIR)


Experiment: vieisl_mska_plus_islr_scratch
Model: mska_plus_islr
Run mode: full
Data dir: /kaggle/input/datasets/leothn/vieisl/skeleton
Metadata: /kaggle/input/datasets/leothn/vieisl/skeleton/dataset_meta.json
Output dir: /kaggle/working/vieisl_mska_plus_islr_scratch


## Dataset and labels

The notebook reads `DATA_DIR/dataset_meta.json` and expects the final VieISL layout with 216 classes, including `ÔN` and a repaired `SUỐI` class. Use `RUN_MODE = "smoke"` for a quick check and `RUN_MODE = "full"` for the complete training run.


In [3]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def save_csv(rows, path: Path, fieldnames=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        keys = set()
        for row in rows:
            keys.update(row.keys())
        fieldnames = sorted(keys)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def load_metadata(path: Path):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing dataset metadata: {path}. Upload the extracted VieISL skeleton folder "
            "with dataset_meta.json, or update META_PATH in the config cell."
        )
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)
    if not isinstance(raw, list):
        raise ValueError("dataset_meta.json must contain a list.")
    meta = []
    for item in raw:
        split = str(item["split"]).lower()
        video = item.get("video") or item.get("video_id")
        gloss = item.get("gloss") or item.get("label")
        npy_path = item.get("npy_path")
        if npy_path is None:
            npy = DATA_DIR / split / f"{video}.npy"
        else:
            npy = DATA_DIR / npy_path
        if not npy.exists():
            raise FileNotFoundError(f"Missing skeleton file for {video}: {npy}")
        meta.append({
            "video": video,
            "split": split,
            "gloss": gloss,
            "npy_path": str(npy),
            "frames": item.get("frames", None),
            "lh_detect_rate": item.get("lh_detect_rate", None),
            "rh_detect_rate": item.get("rh_detect_rate", None),
        })
    return meta


def build_label_maps(meta):
    labels = sorted({item["gloss"] for item in meta})
    label_to_id = {label: idx for idx, label in enumerate(labels)}
    id_to_label = {idx: label for label, idx in label_to_id.items()}
    return label_to_id, id_to_label


class VieISLDataset(Dataset):
    def __init__(self, meta, label_to_id, split: str, augment: bool):
        self.items = [item for item in meta if item["split"] == split]
        self.label_to_id = label_to_id
        self.split = split
        self.augment = augment

    def __len__(self):
        return len(self.items)

    @staticmethod
    def time_resample(arr: np.ndarray, new_len: int):
        if len(arr) == new_len:
            return arr
        idx = np.linspace(0, len(arr) - 1, new_len).astype(np.int64)
        return arr[idx]

    def augment_skeleton(self, sk: np.ndarray):
        if random.random() < CONFIG["time_stretch_prob"]:
            new_len = max(8, int(len(sk) * random.uniform(*CONFIG["time_stretch_range"])))
            sk = self.time_resample(sk, new_len)
        if CONFIG["spatial_jitter_std"] > 0:
            sk[:, :, :2] += np.random.randn(*sk[:, :, :2].shape).astype(np.float32) * CONFIG["spatial_jitter_std"]
        if random.random() < CONFIG["confidence_dropout_prob"]:
            drop = np.random.rand(sk.shape[0], sk.shape[1], 1) < CONFIG["confidence_dropout_ratio"]
            sk[:, :, 2:3] = np.where(drop, 0.0, sk[:, :, 2:3])
        if CONFIG["enable_horizontal_flip"] and random.random() < CONFIG["horizontal_flip_prob"]:
            sk[:, :, 0] = -sk[:, :, 0]
            sk_new = sk.copy()
            sk_new[:, 33:54, :] = sk[:, 54:75, :]
            sk_new[:, 54:75, :] = sk[:, 33:54, :]
            sk = sk_new
        return sk

    def __getitem__(self, idx):
        item = self.items[idx]
        sk = np.load(item["npy_path"], allow_pickle=False).astype(np.float32)
        if sk.ndim != 3 or sk.shape[1] != 86 or sk.shape[2] != 3:
            raise ValueError(f"Expected skeleton shape (T, 86, 3), got {sk.shape}: {item['npy_path']}")
        sk = np.nan_to_num(sk, nan=0.0, posinf=0.0, neginf=0.0)
        if CONFIG["root_normalize"]:
            root_xy = sk[:, 0:1, :2].copy()
            sk[:, :, :2] = sk[:, :, :2] - root_xy
        if self.augment:
            sk = self.augment_skeleton(sk)
        y = self.label_to_id[item["gloss"]]
        return {
            "sk": torch.from_numpy(sk.astype(np.float32)),
            "y": torch.tensor(y, dtype=torch.long),
            "item": item,
            "length": int(sk.shape[0]),
        }


def collate_fn(batch):
    sk = pad_sequence([b["sk"] for b in batch], batch_first=True, padding_value=0.0)
    lengths = torch.tensor([b["length"] for b in batch], dtype=torch.long)
    y = torch.stack([b["y"] for b in batch])
    items = [b["item"] for b in batch]
    return sk, lengths, y, items


def make_sampler(dataset):
    labels = [dataset.label_to_id[item["gloss"]] for item in dataset.items]
    counts = Counter(labels)
    weights = torch.tensor([1.0 / counts[y] for y in labels], dtype=torch.double)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


set_seed(CONFIG["seed"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
meta = load_metadata(META_PATH)
label_to_id, id_to_label = build_label_maps(meta)

split_counts = Counter(item["split"] for item in meta)
label_counts = Counter(item["gloss"] for item in meta if item["split"] == "train")
print("Split counts:", dict(split_counts))
print("Number of classes:", len(label_to_id))
print("Train class count min/median/max:", min(label_counts.values()), np.median(list(label_counts.values())), max(label_counts.values()))

if len(label_to_id) != EXPECTED_CLASSES:
    raise ValueError(f"Expected {EXPECTED_CLASSES} VieISL classes, found {len(label_to_id)}. Check ÔN/SUỐI and metadata.")
if not {"train", "dev", "test"}.issubset(split_counts):
    raise ValueError(f"Expected train/dev/test splits, got {dict(split_counts)}")

save_json(CONFIG, OUTPUT_DIR / "config.json")
save_json(label_to_id, OUTPUT_DIR / "label_to_id.json")
save_json(id_to_label, OUTPUT_DIR / "id_to_label.json")
pd.DataFrame(meta).to_csv(OUTPUT_DIR / "resolved_metadata.csv", index=False)

train_ds = VieISLDataset(meta, label_to_id, "train", augment=True)
dev_ds = VieISLDataset(meta, label_to_id, "dev", augment=False)
test_ds = VieISLDataset(meta, label_to_id, "test", augment=False)

if RUN_MODE == "smoke":
    train_ds = Subset(train_ds, range(min(CONFIG["smoke_train_samples"], len(train_ds))))
    dev_ds = Subset(dev_ds, range(min(CONFIG["smoke_eval_samples"], len(dev_ds))))
    test_ds = Subset(test_ds, range(min(CONFIG["smoke_eval_samples"], len(test_ds))))
    sampler = None
else:
    sampler = make_sampler(train_ds) if CONFIG["use_weighted_sampler"] else None

loader_kwargs = dict(
    batch_size=CONFIG["batch_size"],
    collate_fn=collate_fn,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)
train_loader = DataLoader(train_ds, shuffle=(sampler is None), sampler=sampler, drop_last=False, **loader_kwargs)
dev_loader = DataLoader(dev_ds, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **loader_kwargs)
print(f"Train={len(train_ds)} Dev={len(dev_ds)} Test={len(test_ds)}")


Split counts: {'train': 1614, 'dev': 216, 'test': 216}
Number of classes: 216
Train class count min/median/max: 5 8.0 8
Train=1614 Dev=216 Test=216


## Model definition


In [4]:
MSKA_STREAMS = {
    "body": [0, 11, 12, 13, 14, 15, 16, 23, 24],
    "lhand": list(range(33, 54)),
    "rhand": list(range(54, 75)),
    "face": list(range(75, 86)),
}
INPUT_CHANNELS = 5 if CONFIG["use_motion"] else 3


def add_motion(sk):
    xy = sk[:, :, :, :2]
    dx = torch.cat([torch.zeros_like(xy[:, :1]), xy[:, 1:] - xy[:, :-1]], dim=1)
    return torch.cat([sk, dx.clamp(-1.0, 1.0)], dim=-1)


class SpatialAttentionBlock(nn.Module):
    def __init__(self, d_model: int, n_points: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model)
        self.spatial_bias = nn.Parameter(torch.zeros(n_heads, n_points, n_points))
        self.attn_drop = nn.Dropout(CONFIG["dropout"] * 0.5)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(CONFIG["dropout"] * 0.5),
        )

    def forward(self, x):
        B, T, N, D = x.shape
        def heads(z):
            return z.view(B, T, N, self.n_heads, self.d_head).permute(0, 1, 3, 2, 4).contiguous()
        q = heads(self.q(x))
        k = heads(self.k(x))
        v = heads(self.v(x))
        with torch.amp.autocast("cuda", enabled=False):
            scores = torch.matmul(q.float(), k.float().transpose(-2, -1)) / math.sqrt(self.d_head)
            scores = scores + self.spatial_bias.float().unsqueeze(0).unsqueeze(0)
            attn = torch.softmax(scores, dim=-1)
            attn = self.attn_drop(attn)
            out = torch.matmul(attn, v.float())
        out = out.permute(0, 1, 3, 2, 4).contiguous().view(B, T, N, D)
        x = self.norm1(x + self.o(out.to(x.dtype)))
        x = self.norm2(x + self.ffn(x))
        return x


class MSKAStreamEncoder(nn.Module):
    def __init__(self, n_points: int):
        super().__init__()
        hidden = CONFIG["hidden"]
        self.input_norm = nn.LayerNorm(INPUT_CHANNELS)
        self.proj = nn.Linear(INPUT_CHANNELS, hidden)
        self.point_embed = nn.Parameter(torch.zeros(1, 1, n_points, hidden))
        self.conf_alpha = nn.Parameter(torch.tensor(0.5))
        self.blocks = nn.ModuleList([
            SpatialAttentionBlock(hidden, n_points, CONFIG["num_heads"])
            for _ in range(CONFIG["num_blocks"])
        ])
        self.pool_score = nn.Linear(hidden, 1)
        self.out_norm = nn.LayerNorm(hidden)
        nn.init.trunc_normal_(self.point_embed, std=0.02)

    def forward(self, x):
        confidence = x[..., 2:3].clamp(0.0, 1.0) if x.shape[-1] >= 3 else None
        if CONFIG["use_conf_gate"] and confidence is not None:
            x = x * (1.0 + torch.sigmoid(self.conf_alpha) * confidence)
        h = self.proj(self.input_norm(x)) + self.point_embed
        for block in self.blocks:
            h = block(h)
        scores = self.pool_score(h).squeeze(-1)
        if confidence is not None:
            scores = scores + torch.log(confidence.squeeze(-1).clamp_min(1e-4))
        weights = torch.softmax(scores, dim=-1).unsqueeze(-1)
        return self.out_norm((h * weights).sum(dim=2))


class MultiScaleTemporalBlock(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        kernels = CONFIG["temporal_kernel_sizes"]
        self.norm = nn.LayerNorm(hidden)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(hidden, hidden, kernel_size=k, padding=k // 2, groups=hidden, bias=False),
                nn.BatchNorm1d(hidden),
                nn.GELU(),
            ) for k in kernels
        ])
        self.mix = nn.Sequential(
            nn.Conv1d(hidden * len(kernels), hidden, kernel_size=1, bias=False),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
        )
        self.ffn = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden * 4),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(hidden * 4, hidden),
            nn.Dropout(CONFIG["dropout"] * 0.5),
        )

    def forward(self, x):
        z = self.norm(x).transpose(1, 2)
        z = torch.cat([branch(z) for branch in self.branches], dim=1)
        z = self.mix(z).transpose(1, 2)
        x = x + z
        x = x + self.ffn(x)
        return x


class AttentiveTemporalPool(nn.Module):
    def __init__(self, hidden: int):
        super().__init__()
        self.score = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.Tanh(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, x, lengths):
        scores = self.score(x).squeeze(-1)
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None].to(x.device)
        scores = scores.masked_fill(~mask, -1e4)
        weights = torch.softmax(scores, dim=1)
        return (x * weights.unsqueeze(-1)).sum(dim=1)


class MSKAPlusISLRModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        hidden = CONFIG["hidden"]
        self.enc = nn.ModuleDict({name: MSKAStreamEncoder(len(indices)) for name, indices in MSKA_STREAMS.items()})
        self.stream_gate = nn.Sequential(
            nn.LayerNorm(hidden * 4),
            nn.Linear(hidden * 4, hidden),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(hidden, 4),
        )
        self.fuse = nn.Sequential(
            nn.LayerNorm(hidden * 4),
            nn.Linear(hidden * 4, hidden),
            nn.GELU(),
            nn.Dropout(CONFIG["dropout"]),
        )
        self.temporal = nn.ModuleList([MultiScaleTemporalBlock(hidden) for _ in range(CONFIG["temporal_blocks"])])
        self.rnn = nn.GRU(
            input_size=hidden,
            hidden_size=hidden // 2,
            num_layers=CONFIG["temporal_rnn_layers"],
            batch_first=True,
            bidirectional=True,
            dropout=CONFIG["dropout"] if CONFIG["temporal_rnn_layers"] > 1 else 0.0,
        )
        self.pool = AttentiveTemporalPool(hidden)
        self.cls = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(CONFIG["dropout"]),
            nn.Linear(hidden, num_classes),
        )
        self.transfer_feature_dim = hidden

    def encode_sequence(self, sk, lengths=None):
        sk_in = add_motion(sk) if CONFIG["use_motion"] else sk
        features = [self.enc[name](sk_in[:, :, indices, :]) for name, indices in MSKA_STREAMS.items()]
        concat = torch.cat(features, dim=-1)
        gates = torch.softmax(self.stream_gate(concat), dim=-1)
        gated = torch.cat([feat * gates[:, :, i:i + 1] for i, feat in enumerate(features)], dim=-1)
        z = self.fuse(gated)
        for block in self.temporal:
            z = block(z)
        self.rnn.flatten_parameters()
        z, _ = self.rnn(z)
        return z

    def encode(self, sk, lengths=None):
        z = self.encode_sequence(sk, lengths)
        if lengths is None:
            return z.mean(dim=1)
        return self.pool(z, lengths)

    def forward(self, sk, lengths=None):
        return self.cls(self.encode(sk, lengths))


def create_model(num_classes: int):
    return MSKAPlusISLRModel(num_classes)


## Training and evaluation utilities


In [5]:
def class_weights_from_train(meta, label_to_id):
    train_labels = [label_to_id[item["gloss"]] for item in meta if item["split"] == "train"]
    counts = Counter(train_labels)
    weights = torch.ones(len(label_to_id), dtype=torch.float32)
    for label_id in range(len(label_to_id)):
        weights[label_id] = 1.0 / max(1, counts[label_id])
    weights = weights / weights.mean()
    return weights


def topk_accuracy(logits, y, k: int):
    k = min(k, logits.shape[1])
    pred = logits.topk(k, dim=1).indices
    return (pred == y[:, None]).any(dim=1).float().mean().item()


def macro_f1(y_true, y_pred, num_classes):
    f1s = []
    for c in range(num_classes):
        tp = sum((yt == c and yp == c) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != c and yp == c) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == c and yp != c) for yt, yp in zip(y_true, y_pred))
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        if precision + recall == 0:
            f1s.append(0.0)
        else:
            f1s.append(2 * precision * recall / (precision + recall))
    return float(np.mean(f1s))


@torch.no_grad()
def evaluate(model, loader, id_to_label, split_name):
    model.eval()
    total = 0
    loss_sum = 0.0
    top1_sum = 0.0
    top5_sum = 0.0
    y_true = []
    y_pred = []
    rows = []
    for sk, lengths, y, items in loader:
        sk = sk.to(DEVICE, non_blocking=True)
        lengths = lengths.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        logits = model(sk, lengths)
        loss = F.cross_entropy(logits, y)
        probs = torch.softmax(logits, dim=1)
        pred = logits.argmax(dim=1)
        bs = y.size(0)
        total += bs
        loss_sum += loss.item() * bs
        top1_sum += (pred == y).float().sum().item()
        top5_sum += topk_accuracy(logits, y, 5) * bs
        y_true.extend(y.cpu().tolist())
        y_pred.extend(pred.cpu().tolist())
        top5 = probs.topk(min(5, probs.shape[1]), dim=1)
        for i, item in enumerate(items):
            rows.append({
                "split": split_name,
                "video": item["video"],
                "reference": id_to_label[int(y[i].cpu())],
                "prediction": id_to_label[int(pred[i].cpu())],
                "correct": int(pred[i].cpu()) == int(y[i].cpu()),
                "confidence": float(probs[i, pred[i]].cpu()),
                "top5": " ".join(id_to_label[int(idx)] for idx in top5.indices[i].cpu().tolist()),
            })
    return {
        "split": split_name,
        "loss": loss_sum / max(1, total),
        "top1": top1_sum / max(1, total),
        "top5": top5_sum / max(1, total),
        "macro_f1": macro_f1(y_true, y_pred, len(id_to_label)),
        "samples": total,
    }, rows, y_true, y_pred


def save_checkpoint(path, model, optimizer, scheduler, epoch, best_dev_top1, history):
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "best_dev_top1": best_dev_top1,
        "history": history,
        "label_to_id": label_to_id,
        "id_to_label": id_to_label,
        "config": CONFIG,
        "experiment_name": EXPERIMENT_NAME,
    }, path)


def train_model(model, train_loader, dev_loader):
    class_weights = class_weights_from_train(meta, label_to_id).to(DEVICE) if CONFIG["use_class_weights"] else None
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])

    def lr_lambda(epoch):
        if epoch < CONFIG["warmup_epochs"]:
            return (epoch + 1) / max(1, CONFIG["warmup_epochs"])
        progress = (epoch - CONFIG["warmup_epochs"]) / max(1, CONFIG["epochs"] - CONFIG["warmup_epochs"])
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    history = []
    best_dev_top1 = -1.0
    patience = 0

    for epoch in range(CONFIG["epochs"]):
        model.train()
        running = 0.0
        seen = 0
        pbar = tqdm(train_loader, desc=f"epoch {epoch:03d}", leave=False)
        for sk, lengths, y, _ in pbar:
            sk = sk.to(DEVICE, non_blocking=True)
            lengths = lengths.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(sk, lengths)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            running += loss.item() * y.size(0)
            seen += y.size(0)
            pbar.set_postfix(loss=running / max(1, seen))
        scheduler.step()
        dev_metrics, _, _, _ = evaluate(model, dev_loader, id_to_label, "dev")
        record = {
            "epoch": epoch,
            "train_loss": running / max(1, seen),
            "dev_loss": dev_metrics["loss"],
            "dev_top1": dev_metrics["top1"],
            "dev_top5": dev_metrics["top5"],
            "dev_macro_f1": dev_metrics["macro_f1"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(record)
        pd.DataFrame(history).to_csv(OUTPUT_DIR / "training_log.csv", index=False)
        print(
            f"Epoch {epoch:03d} | loss={record['train_loss']:.4f} | "
            f"dev top1={record['dev_top1']:.4f} | dev top5={record['dev_top5']:.4f} | "
            f"macroF1={record['dev_macro_f1']:.4f}"
        )
        save_checkpoint(OUTPUT_DIR / "last_model.pt", model, optimizer, scheduler, epoch, best_dev_top1, history)
        if dev_metrics["top1"] > best_dev_top1:
            best_dev_top1 = dev_metrics["top1"]
            patience = 0
            save_checkpoint(OUTPUT_DIR / "best_model.pt", model, optimizer, scheduler, epoch, best_dev_top1, history)
            print(f"Saved best model with Dev Top-1={best_dev_top1:.4f}")
        else:
            patience += 1
            if patience >= CONFIG["patience"]:
                print(f"Early stopping at epoch {epoch}. Best Dev Top-1={best_dev_top1:.4f}")
                break
    return history


## Plotting utilities


In [6]:
def plot_history(history):
    if not history:
        return
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(df["epoch"], df["train_loss"], marker="o", label="train")
    axes[0].plot(df["epoch"], df["dev_loss"], marker="o", label="dev")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[1].plot(df["epoch"], df["dev_top1"] * 100, marker="o", label="Top-1")
    axes[1].plot(df["epoch"], df["dev_top5"] * 100, marker="o", label="Top-5")
    axes[1].set_title("Dev Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy (%)")
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    axes[2].plot(df["epoch"], df["dev_macro_f1"] * 100, marker="o", color="tab:green")
    axes[2].set_title("Dev Macro-F1")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Macro-F1 (%)")
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=220)
    plt.close()


def confusion_matrix_np(y_true, y_pred, num_classes):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def save_confusion_outputs(y_true, y_pred, id_to_label, prefix):
    cm = confusion_matrix_np(y_true, y_pred, len(id_to_label))
    np.save(OUTPUT_DIR / f"{prefix}_confusion_matrix.npy", cm)
    confusions = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if i != j and cm[i, j] > 0:
                confusions.append({
                    "reference": id_to_label[i],
                    "prediction": id_to_label[j],
                    "count": int(cm[i, j]),
                })
    confusions = sorted(confusions, key=lambda r: -r["count"])
    save_csv(confusions[:100], OUTPUT_DIR / f"{prefix}_top_confusions.csv", ["reference", "prediction", "count"])

    per_class = []
    for i in range(cm.shape[0]):
        total = cm[i].sum()
        correct = cm[i, i]
        per_class.append({
            "gloss": id_to_label[i],
            "samples": int(total),
            "correct": int(correct),
            "accuracy": float(correct / max(1, total)),
        })
    save_csv(per_class, OUTPUT_DIR / f"{prefix}_per_class_accuracy.csv", ["gloss", "samples", "correct", "accuracy"])

    top_labels = sorted(range(cm.shape[0]), key=lambda i: cm[i].sum(), reverse=True)[:30]
    sub = cm[np.ix_(top_labels, top_labels)]
    labels = [id_to_label[i] for i in top_labels]
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(sub, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_title(f"{prefix.title()} Confusion Matrix, Top 30 Classes")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"{prefix}_confusion_matrix_top30.png", dpi=220)
    plt.close()


def plot_class_distribution(meta):
    rows = []
    for split in ["train", "dev", "test"]:
        counts = Counter(item["gloss"] for item in meta if item["split"] == split)
        for gloss, count in counts.items():
            rows.append({"split": split, "gloss": gloss, "count": count})
    pd.DataFrame(rows).to_csv(OUTPUT_DIR / "class_distribution.csv", index=False)
    train_counts = [r["count"] for r in rows if r["split"] == "train"]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(train_counts, bins=range(1, max(train_counts) + 2), color="#4C78A8", edgecolor="white")
    ax.set_title("VieISL Train Samples per Gloss")
    ax.set_xlabel("Samples per gloss")
    ax.set_ylabel("Number of glosses")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "class_distribution_train.png", dpi=220)
    plt.close()


## Run experiment

The best checkpoint is selected by development Top-1 accuracy. Final reporting uses the held-out test split.


In [7]:
plot_class_distribution(meta)
model = create_model(len(label_to_id)).to(DEVICE)
num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model summary")
print(f"  Experiment           : {EXPERIMENT_NAME}")
print(f"  Model                : {model.__class__.__name__}")
print(f"  Dataset              : {DATASET_NAME}")
print(f"  Task                 : {TASK_NAME}")
print(f"  Run mode             : {RUN_MODE}")
print(f"  Classes              : {len(label_to_id)}")
print(f"  Parameters           : {num_params:,}")
print(f"  Trainable parameters : {trainable_params:,}")

start_time = time.time()
history = train_model(model, train_loader, dev_loader)
plot_history(history)

best_path = OUTPUT_DIR / "best_model.pt"
best_epoch = None
best_dev_top1 = None
if best_path.exists():
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    best_epoch = ckpt.get("epoch")
    best_dev_top1 = ckpt.get("best_dev_top1")
    print(f"Loaded best checkpoint from epoch {best_epoch} with Dev Top-1={best_dev_top1:.4f}")

dev_metrics, dev_rows, dev_y_true, dev_y_pred = evaluate(model, dev_loader, id_to_label, "dev")
test_metrics, test_rows, test_y_true, test_y_pred = evaluate(model, test_loader, id_to_label, "test")
save_csv(dev_rows, OUTPUT_DIR / "predictions_dev.csv")
save_csv(test_rows, OUTPUT_DIR / "predictions_test.csv")
save_confusion_outputs(test_y_true, test_y_pred, id_to_label, "test")

final_metrics = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "dataset_name": DATASET_NAME,
    "task": TASK_NAME,
    "run_mode": RUN_MODE,
    "classes": len(label_to_id),
    "parameters_total": int(num_params),
    "parameters_trainable_initial": int(trainable_params),
    "elapsed_minutes": (time.time() - start_time) / 60.0,
    "best_epoch": best_epoch,
    "best_dev_top1": best_dev_top1,
    "transfer_feature_dim": getattr(model, "transfer_feature_dim", None),
    "dev": dev_metrics,
    "test": test_metrics,
}
save_json(final_metrics, OUTPUT_DIR / "metrics.json")

summary = pd.DataFrame([
    {
        "Split": "Dev",
        "Top-1 (%)": round(dev_metrics["top1"] * 100, 2),
        "Top-5 (%)": round(dev_metrics["top5"] * 100, 2),
        "Macro-F1 (%)": round(dev_metrics["macro_f1"] * 100, 2),
        "Loss": round(dev_metrics["loss"], 4),
        "Samples": dev_metrics["samples"],
    },
    {
        "Split": "Test",
        "Top-1 (%)": round(test_metrics["top1"] * 100, 2),
        "Top-5 (%)": round(test_metrics["top5"] * 100, 2),
        "Macro-F1 (%)": round(test_metrics["macro_f1"] * 100, 2),
        "Loss": round(test_metrics["loss"], 4),
        "Samples": test_metrics["samples"],
    },
])
summary.to_csv(OUTPUT_DIR / "final_metrics_summary.csv", index=False)
print("Final evaluation summary")
display(summary)

torch.save({
    "encoder_state": {k: v for k, v in model.state_dict().items() if not k.startswith("cls.")},
    "label_to_id": label_to_id,
    "id_to_label": id_to_label,
    "config": CONFIG,
    "model_name": MODEL_NAME,
    "transfer_feature_dim": getattr(model, "transfer_feature_dim", None),
    "transfer_note": "For VieCSL transfer, reuse the encoder_state and replace the ISLR pooling/classification head with a CTC temporal head.",
}, OUTPUT_DIR / "encoder_state_for_transfer.pt")

save_json({
    "source_task": "VieISL isolated sign recognition",
    "target_task": "VieCSL continuous sign recognition",
    "model_name": MODEL_NAME,
    "transfer_feature_dim": getattr(model, "transfer_feature_dim", None),
    "recommended_transfer": "Load encoder_state_for_transfer.pt, initialize the matching encoder, remove pooling/classification, and attach a CTC head for gloss-sequence prediction.",
}, OUTPUT_DIR / "transfer_config.json")

print("Artifacts saved to:", OUTPUT_DIR)


Model summary
  Experiment           : vieisl_mska_plus_islr_scratch
  Model                : MSKAPlusISLRModel
  Dataset              : VieISL
  Task                 : isolated_sign_recognition
  Run mode             : full
  Classes              : 216
  Parameters           : 16,082,829
  Trainable parameters : 16,082,829


epoch 000:   0%|          | 0/162 [00:00<?, ?it/s]

Epoch 000 | loss=5.5920 | dev top1=0.0139 | dev top5=0.0417 | macroF1=0.0022
Saved best model with Dev Top-1=0.0139


epoch 001:   0%|          | 0/162 [00:00<?, ?it/s]

Epoch 001 | loss=5.3271 | dev top1=0.0509 | dev top5=0.1389 | macroF1=0.0128
Saved best model with Dev Top-1=0.0509


...

Epoch 139 | loss=0.8998 | dev top1=0.9769 | dev top5=1.0000 | macroF1=0.9691


epoch 140:   0%|          | 0/162 [00:00<?, ?it/s]

Epoch 140 | loss=0.8990 | dev top1=0.9815 | dev top5=1.0000 | macroF1=0.9753
Early stopping at epoch 140. Best Dev Top-1=0.9861
Loaded best checkpoint from epoch 108 with Dev Top-1=0.9861
Final evaluation summary


,Split,Top-1 (%),Top-5 (%),Macro-F1 (%),Loss,Samples
0,Dev,98.61,100.00,98.15,0.1983,216
1,Test,93.98,99.07,93.06,0.3138,216


Artifacts saved to: /kaggle/working/vieisl_mska_plus_islr_scratch
